In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import ipdb

### Transformer model

In [59]:
class input_embedding(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(input_embedding, self).__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)

    def forward(self, x):
        x = self.fc(x)
        return x

class BBox2Activity(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim, num_heads, num_layers, dropout):
        super(BBox2Activity, self).__init__()

        # TODO: Implement positional encoding for frame number (and encoding for bbox?)
        self.encoder_embedding = input_embedding(input_dim, hidden_dim)
        self.encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(\
            hidden_dim, num_heads, hidden_dim, dropout, batch_first=True), num_layers)
        self.decoder_embedding = nn.Embedding(1, 256)
        self.decoder = nn.TransformerDecoder(nn.TransformerDecoderLayer(\
            hidden_dim, num_heads, hidden_dim, dropout, batch_first=True), num_layers)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, src, tgt, src_padding_mask):
        # ipdb.set_trace()
        encoder_embedded = self.encoder_embedding(src)
        encoder_outputs = self.encoder(encoder_embedded, src_key_padding_mask=src_padding_mask)

        decoder_embedded = self.decoder_embedding(tgt)
        decoder_outputs = self.decoder(decoder_embedded, encoder_outputs)

        predictions = self.fc(decoder_outputs)
        return predictions


In [60]:
ip = torch.rand(64, 1142, 289)
mask = torch.rand(64, 1142)
tgt = torch.zeros([64, 1], dtype=torch.int)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define hyperparameters
input_dim = 289  # frame encoding size + bbox encoding size + bbox label size
output_dim = 91  # activity class size
hidden_dim = 256
num_heads = 4
num_layers = 2
dropout = 0.1
learning_rate = 0.001
# batch_size = 64
epochs = 100

# Create the model instance
model = BBox2Activity(input_dim, output_dim, hidden_dim, num_heads, num_layers, dropout).to(device)


In [61]:
op = model(ip, tgt, mask)

In [62]:
op.size()

torch.Size([64, 1, 91])

In [48]:
embedding = nn.Embedding(1, 256)
print(embedding(torch.LongTensor([0])).size())

torch.Size([1, 256])


In [64]:
bbox_mask = torch.ones(20)
bbox_mask[:5] = 0
print(bbox_mask)
print(bbox_mask.to(torch.bool))

tensor([0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1.])
tensor([False, False, False, False, False,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,  True])
